# 🕵️ Terra · Detective de Tendencias del Sistema Tierra
### NASA Space Apps Challenge 2026 — *Be an Earth System Trend Detective!*

**¿Está cambiando el planeta donde vives?** Esta herramienta te convierte en detective de datos de la NASA:

1. **🔎 Elige una pista:** ¿qué quieres investigar (calor, lluvia, sequía, contaminación, nieve, océano…) y **dónde**?
2. **🛰️ Consigue la evidencia:** la herramienta arma el enlace a **NASA Giovanni** con tu variable y región; descargas la serie de tiempo (CSV) y la subes aquí.
3. **📈 Analiza la escena:** vemos cómo cambia la variable en el tiempo, quitamos el ciclo de las estaciones y medimos **cuánto cambia por década**.
4. **⚖️ Da tu veredicto:** con pruebas estadísticas (**regresión con prueba *t*, intervalo de confianza 95 % y Mann-Kendall**) decidimos si el cambio es **significativo** o podría ser casualidad.
5. **🗺️ Compara escenas:** la misma variable puede **subir en una región y bajar en otra** por el mismo proceso.

> **Cómo usarlo:** menú *Entorno de ejecución → Ejecutar todo*. Primero corren **3 casos resueltos** (no necesitas cuenta). Después, en la sección **"Tu investigación"**, eliges variable y región.
>
> *English: Run all cells. Three solved demo cases run first; then pick any NASA variable and region in "Your investigation".*

In [ ]:
#@title ⚙️ Preparar herramientas (ejecuta primero) { display-mode: "form" }
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymannkendall"], check=False)
import io, re, textwrap, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import pymannkendall as mk
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 10})
print("✅ Listo")

## 📚 1. El archivo del detective: variables de la NASA
Seleccionamos variables de **satélites** (instrumentos que miden desde el espacio) y de **modelos** (simulaciones que combinan satélites, estaciones y física). Todas están en **NASA Giovanni** y se pueden promediar sobre cualquier región.

In [ ]:
#@title 📚 Catálogo de variables (con definiciones simples) { display-mode: "form" }
# conv: función para pasar a unidades fáciles de entender
K2C   = (lambda v: v - 273.15, "°C")
FLUX  = (lambda v: v * 86400, "mm/día")          # kg m-2 s-1 -> mm/día
MMHR  = (lambda v: v * 24, "mm/día")             # mm/h -> mm/día
NO2   = (lambda v: v / 1e15, "×10¹⁵ moléc/cm²")
QKG   = (lambda v: v * 1000, "g/kg")
SAME  = lambda u: (lambda v: v, u)

CATALOGO = {
 # tema, nombre, definición sencilla, fuente, conversión, desde
 "MERRA2_T2M":   dict(id="M2TMNXSLV_5_12_4_T2M", tema="🌡️ Calor", nombre="Temperatura del aire a 2 m",
     defin="Qué tan caliente está el aire a la altura de una persona.", fuente="Modelo MERRA-2 (NASA GMAO), 0.5°", conv=K2C, desde="1980"),
 "AIRS_TAIR":    dict(id="AIRS3STM_7_0_SurfAirTemp_A", tema="🌡️ Calor", nombre="Temperatura del aire en superficie (de día)",
     defin="Temperatura del aire medida por el satélite Aqua al pasar de día.", fuente="Satélite Aqua · instrumento AIRS, 1°", conv=K2C, desde="2002"),
 "GLDAS_TSKIN":  dict(id="GLDAS_NOAH025_M_2_1_AvgSurfT_inst", tema="🌡️ Calor", nombre="Temperatura de la superficie del suelo",
     defin="Qué tan caliente está la 'piel' del terreno (más que el aire en días soleados).", fuente="Modelo GLDAS-2.1, 0.25°", conv=K2C, desde="2000"),
 "GPM_PRECIP":   dict(id="GPM_3IMERGM_07_precipitation", tema="🌧️ Lluvia", nombre="Lluvia (satélites GPM)",
     defin="Cuánta agua cae del cielo, estimada por una constelación de satélites.", fuente="Misión GPM IMERG, 0.1°", conv=MMHR, desde="1998"),
 "MERRA2_PRECIP":dict(id="M2TMNXFLX_5_12_4_PRECTOTCORR", tema="🌧️ Lluvia", nombre="Lluvia (modelo, corregida)",
     defin="Precipitación total del modelo, ajustada con observaciones.", fuente="Modelo MERRA-2, 0.5°", conv=FLUX, desde="1980"),
 "FLDAS_SM":     dict(id="FLDAS_NOAH01_C_GL_M_001_SoilMoi10_40cm_tavg", tema="🏜️ Sequía y agua", nombre="Humedad del suelo (10–40 cm)",
     defin="Fracción del suelo ocupada por agua donde están las raíces. Baja = sequía.", fuente="Modelo FLDAS (NASA/USAID), 0.1°", conv=SAME("m³/m³"), desde="1982"),
 "FLDAS_ET":     dict(id="FLDAS_NOAH01_C_GL_M_001_Evap_tavg", tema="🏜️ Sequía y agua", nombre="Evapotranspiración",
     defin="Agua que regresa al aire desde el suelo y las plantas ('sudor' del paisaje).", fuente="Modelo FLDAS, 0.1°", conv=FLUX, desde="1982"),
 "FLDAS_RUNOFF": dict(id="FLDAS_NOAH01_C_GL_M_001_Qs_tavg", tema="🏜️ Sequía y agua", nombre="Escurrimiento superficial",
     defin="Agua de lluvia que corre por la superficie hacia ríos (no se infiltra).", fuente="Modelo FLDAS, 0.1°", conv=FLUX, desde="1982"),
 "MERRA2_QV2M":  dict(id="M2TMNXSLV_5_12_4_QV2M", tema="💧 Humedad del aire", nombre="Humedad específica a 2 m",
     defin="Gramos de vapor de agua por kilo de aire. Aire más cálido puede cargar más.", fuente="Modelo MERRA-2, 0.5°", conv=QKG, desde="1980"),
 "OMI_NO2":      dict(id="OMNO2d_004_ColumnAmountNO2TropCloudScreened", tema="🏭 Calidad del aire", nombre="Dióxido de nitrógeno (NO₂)",
     defin="Gas contaminante de autos, industria y plantas eléctricas, medido desde el espacio.", fuente="Satélite Aura · instrumento OMI, 0.25° (diario)", conv=NO2, desde="2004"),
 "MODIS_AOD":    dict(id="MOD08_M3_6_1_AOD_550_Dark_Target_Deep_Blue_Combined_Mean_Mean", tema="🏭 Calidad del aire", nombre="Aerosoles (polvo, humo, smog)",
     defin="Qué tan 'turbia' está la atmósfera por partículas. 0 = cielo limpio, >1 = muy contaminado.", fuente="Satélite Terra · instrumento MODIS, 1°", conv=SAME("adimensional"), desde="2000"),
 "MERRA2_CO":    dict(id="M2TMNXCHM_5_12_4_COSC", tema="🏭 Calidad del aire", nombre="Monóxido de carbono en superficie",
     defin="Gas de la combustión incompleta (incendios, motores).", fuente="Modelo MERRA-2, 0.5°", conv=SAME("ppbv"), desde="1980"),
 "MODIS_CLOUD":  dict(id="MOD08_M3_6_1_Cloud_Fraction_Mean_Mean", tema="☁️ Nubes y sol", nombre="Fracción de nubes",
     defin="Porción del cielo cubierta de nubes (0 = despejado, 1 = nublado).", fuente="Satélite Terra · MODIS, 1°", conv=SAME("fracción"), desde="2000"),
 "MERRA2_SW":    dict(id="M2TMNXRAD_5_12_4_SWGDN", tema="☁️ Nubes y sol", nombre="Radiación solar en superficie",
     defin="Energía del sol que llega al suelo (útil para paneles solares).", fuente="Modelo MERRA-2, 0.5°", conv=SAME("W/m²"), desde="1980"),
 "MERRA2_O3":    dict(id="M2TMNXCHM_5_12_4_TO3", tema="🛡️ Ozono", nombre="Ozono total en la columna",
     defin="Cantidad de ozono sobre ti; la capa que filtra los rayos UV.", fuente="Modelo MERRA-2, 0.5°", conv=SAME("unidades Dobson"), desde="1980"),
 "FLDAS_SNOW":   dict(id="FLDAS_NOAH01_C_GL_M_001_SnowCover_inst", tema="❄️ Nieve", nombre="Cobertura de nieve",
     defin="Fracción del terreno cubierta de nieve.", fuente="Modelo FLDAS, 0.1°", conv=SAME("fracción"), desde="1982"),
 "MERRA2_SST":   dict(id="M2TMNXOCN_5_12_4_TSKINWTR", tema="🌊 Océano", nombre="Temperatura del mar (superficie)",
     defin="Qué tan caliente está la superficie del océano.", fuente="Modelo MERRA-2, 0.5°", conv=K2C, desde="1980"),
 "MODIS_CHL":    dict(id="MODISA_L3m_CHL_Mo_4km_R2022_0_chlor_a", tema="🌊 Océano", nombre="Clorofila (fitoplancton)",
     defin="'Verdor' del mar: algas microscópicas que son la base de la vida marina.", fuente="Satélite Aqua · MODIS, 4 km", conv=SAME("mg/m³"), desde="2002"),
}
POR_ID = {v["id"]: k for k, v in CATALOGO.items()}
tabla = pd.DataFrame([{ "Tema": v["tema"], "Variable": v["nombre"], "¿Qué es?": v["defin"], "Fuente": v["fuente"], "Desde": v["desde"]} for v in CATALOGO.values()])
display(tabla.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## 🔬 2. El laboratorio: cómo decide el detective
Para cada serie de tiempo:
- **Anomalías:** restamos el promedio de cada mes (todos los eneros, todos los febreros…). Así el ciclo de las estaciones no esconde la tendencia.
- **Regresión lineal + prueba *t*:** la pendiente es la *velocidad del cambio*. Corregimos porque los meses consecutivos se parecen entre sí (autocorrelación), lo que si no se corrige exagera la confianza.
- **Intervalo de confianza 95 %:** el rango donde probablemente está la verdadera velocidad de cambio. Si **no incluye el cero**, la tendencia es significativa.
- **Mann-Kendall estacional + pendiente de Sen:** prueba que no supone una línea recta perfecta y resiste valores extremos.
- **¿Cuántos años faltan?** Si no es significativa, estimamos cuántos años de datos harían falta para detectarla (Weatherhead et al., 1998).

In [ ]:
#@title 🔬 Motor de análisis (funciones) { display-mode: "form" }
def leer_giovanni(texto_o_ruta):
    """Lee el CSV 'Time Series, Area-Averaged' de Giovanni (o un CSV fecha,valor)."""
    txt = open(texto_o_ruta, encoding="utf-8", errors="ignore").read() if not ("\n" in str(texto_o_ruta)) else texto_o_ruta
    lineas = txt.splitlines()
    meta = {}
    for l in lineas[:15]:
        if ":," in l:
            k, v = l.split(":,", 1); meta[k.strip()] = v.strip().strip('"')
    i0 = next((i for i, l in enumerate(lineas) if l.lower().startswith(("time,", "fecha,"))), None)
    df = pd.read_csv(io.StringIO("\n".join(lineas[i0:])), skipinitialspace=True)
    col_t, col_v = df.columns[0], df.columns[1]
    s = pd.Series(pd.to_numeric(df[col_v], errors="coerce").values, index=pd.to_datetime(df[col_t])).replace(-9999, np.nan)
    if s.index.to_series().diff().median() < pd.Timedelta(days=20):   # diaria -> mensual
        s = s.resample("MS").mean()
    s = s.asfreq("MS")
    var_id = col_v.replace("mean_", "").strip()
    return s, var_id, meta

def preparar(s, var_id):
    k = POR_ID.get(var_id)
    if k:
        f, u = CATALOGO[k]["conv"]; return f(s), CATALOGO[k]["nombre"], u, CATALOGO[k]
    return s, var_id, "unidades originales", None

def _p_texto(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def analizar(s, nombre, unidad, region="", mostrar=True, color="#1f6fb2"):
    s = s.dropna()
    clim = s.groupby(s.index.month).transform("mean")
    anom = (s - clim)
    t = (anom.index.year + (anom.index.month - 0.5) / 12).values
    y = anom.values; n = len(y)
    # regresión OLS
    b, a = np.polyfit(t, y, 1)
    res = y - (a + b * t)
    r1 = np.corrcoef(res[:-1], res[1:])[0, 1]                 # autocorrelación lag-1
    n_ef = max(3, n * (1 - r1) / (1 + r1))                    # tamaño efectivo (Santer et al. 2000)
    se = np.sqrt(np.sum(res**2) / (n_ef - 2) / np.sum((t - t.mean())**2))
    tstat = b / se; p_t = 2 * stats.t.sf(abs(tstat), n_ef - 2)
    tcrit = stats.t.ppf(0.975, n_ef - 2); ci = (b - tcrit * se, b + tcrit * se)
    # Mann-Kendall estacional
    m = mk.seasonal_test(s.values, period=12)
    sen_dec = m.slope * 10
    # años necesarios para detectar (Weatherhead 1998)
    sigma = np.std(res); years = (len(s) / 12)
    n_need = ((3.3 * sigma / max(abs(b), 1e-12)) * np.sqrt((1 + r1) / (1 - r1))) ** (2 / 3) if abs(b) > 0 else np.inf
    r = dict(nombre=nombre, unidad=unidad, region=region, inicio=s.index[0], fin=s.index[-1], anios=years,
             pend_dec=b * 10, ic_dec=(ci[0] * 10, ci[1] * 10), p_t=p_t, p_mk=m.p, tend_mk=m.trend, sen_dec=sen_dec,
             cambio_total=b * years, media=s.mean(), r1=r1, anios_necesarios=n_need, s=s, anom=anom, a=a, b=b)
    if mostrar:
        graficar(r, color); veredicto(r)
    return r

def graficar(r, color="#1f6fb2"):
    s, anom = r["s"], r["anom"]
    fig, ax = plt.subplots(2, 1, figsize=(12, 6.2), sharex=True, gridspec_kw={"height_ratios": [1, 1.1]})
    ax[0].plot(s.index, s.values, color=color, lw=.9)
    ax[0].plot(s.index, s.rolling(12, center=True).mean(), color="k", lw=1.6, label="promedio móvil 12 meses")
    ax[0].set_ylabel(r["unidad"]); ax[0].legend(loc="upper left", fontsize=8)
    ax[0].set_title(f"{r['nombre']} · {r['region']}  ({r['inicio']:%Y}–{r['fin']:%Y})", loc="left", fontweight="bold")
    ax[1].bar(anom.index, anom.values, width=25, color=np.where(anom.values >= 0, "#d9534f", "#2c7fb8"), alpha=.75)
    t = (anom.index.year + (anom.index.month - .5) / 12).values
    lin = r["a"] + r["b"] * t
    lo = r["a"] + r["ic_dec"][0] / 10 * (t - t.mean()) + r["b"] * t.mean()
    hi = r["a"] + r["ic_dec"][1] / 10 * (t - t.mean()) + r["b"] * t.mean()
    ax[1].fill_between(anom.index, lo, hi, color="k", alpha=.12, label="intervalo de confianza 95 %")
    ax[1].plot(anom.index, lin, "k-", lw=2, label=f"tendencia: {r['pend_dec']:+.3g} {r['unidad']}/década")
    ax[1].axhline(0, color="gray", lw=.8)
    ax[1].set_ylabel("anomalía\n(sin estaciones)"); ax[1].legend(loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()

def veredicto(r):
    sig = r["p_t"] < 0.05 and r["p_mk"] < 0.05
    sube = r["pend_dec"] > 0
    flecha = "⬆️ AUMENTA" if sube else "⬇️ DISMINUYE"
    if r["p_t"] < 0.01 and r["p_mk"] < 0.01: nivel, emo = "MUY SIGNIFICATIVA", "🟥"
    elif sig: nivel, emo = "SIGNIFICATIVA", "🟧"
    elif min(r["p_t"], r["p_mk"]) < 0.10: nivel, emo = "INDICIO (aún no concluyente)", "🟨"
    else: nivel, emo = "NO SIGNIFICATIVA", "⬜"
    u = r["unidad"]
    rel = 100 * r["cambio_total"] / abs(r["media"]) if r["media"] else np.nan
    txt = f"""
### {emo} Veredicto: {flecha} · tendencia **{nivel}**
| | |
|---|---|
| **Velocidad de cambio** | **{r['pend_dec']:+.3g} {u} por década** (IC 95 %: {r['ic_dec'][0]:+.3g} a {r['ic_dec'][1]:+.3g}) |
| **Cambio acumulado** en {r['anios']:.0f} años | {r['cambio_total']:+.3g} {u}  (≈ {rel:+.1f} % del promedio {r['media']:.3g} {u}) |
| Prueba *t* (corregida por autocorrelación) | p = {_p_texto(r['p_t'])} |
| Mann-Kendall estacional | p = {_p_texto(r['p_mk'])} · pendiente de Sen {r['sen_dec']:+.3g} {u}/década |
"""
    if sig:
        txt += f"\n🗣️ **En palabras simples:** el cambio es real con alta confianza: si en realidad no hubiera tendencia, ver un patrón así por pura casualidad ocurriría menos de {max(1, round(100*max(r['p_t'], r['p_mk'])))} de cada 100 veces."
    else:
        extra = r["anios_necesarios"] - r["anios"]
        txt += f"\n🗣️ **En palabras simples:** con estos datos no podemos descartar que sea casualidad: el 'ruido' de un año a otro es más grande que la tendencia."
        if np.isfinite(extra) and extra > 0 and extra < 200:
            txt += f" Si la tendencia actual continúa, se necesitarían **≈{r['anios_necesarios']:.0f} años** de datos para detectarla (tenemos {r['anios']:.0f}). *Una tendencia no significativa también es un hallazgo.*"
    display(Markdown(txt))

def comparar(r1, r2):
    """Compara la misma variable en dos regiones (o dos variables)."""
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1]})
    for r, c in [(r1, "#2c7fb8"), (r2, "#d9534f")]:
        z = r["anom"] / r["anom"].std()
        ax[0].plot(z.index, z.rolling(12, center=True).mean(), color=c, lw=2, label=f"{r['region']}")
    ax[0].axhline(0, color="gray", lw=.8); ax[0].set_ylabel("anomalía estandarizada\n(promedio 12 meses)")
    ax[0].set_title(f"{r1['nombre']}: dos regiones, ¿misma dirección?", loc="left", fontweight="bold"); ax[0].legend()
    for i, (r, c) in enumerate([(r1, "#2c7fb8"), (r2, "#d9534f")]):
        rel = lambda v: 100 * v / abs(r["media"])
        ax[1].errorbar(i, rel(r["pend_dec"]), yerr=[[rel(r["pend_dec"]) - rel(r["ic_dec"][0])], [rel(r["ic_dec"][1]) - rel(r["pend_dec"])]],
                       fmt="o", color=c, capsize=8, ms=9)
    ax[1].axhline(0, color="k", lw=1); ax[1].set_xticks([0, 1]); ax[1].set_xticklabels([r1["region"], r2["region"]], fontsize=8)
    ax[1].set_ylabel("% de cambio por década"); ax[1].set_title("Velocidad ± IC 95 %", loc="left"); ax[1].set_xlim(-.6, 1.6)
    plt.tight_layout(); plt.show()

def enlace_giovanni(var_key, bbox, inicio, fin):
    v = CATALOGO[var_key]["id"]
    url = (f"https://giovanni.gsfc.nasa.gov/giovanni/#service=ArAvTs&starttime={inicio}T00:00:00Z"
           f"&endtime={fin}T23:59:59Z&bbox={bbox}&data={v}")
    return url
print("✅ Motor cargado")

In [ ]:
#@title 🗂️ Datos de los casos resueltos (descargados de NASA Giovanni) { display-mode: "form" }
DEMOS = {"morelos_lluvia": "fecha,valor\n2000-01-01,0.066\n2000-02-01,0.1\n2000-03-01,0.105\n2000-04-01,0.227\n2000-05-01,3.503\n2000-06-01,8.311\n2000-07-01,3.796\n2000-08-01,6.233\n2000-09-01,6.076\n2000-10-01,1.701\n2000-11-01,0.155\n2000-12-01,0.112\n2001-01-01,0.287\n2001-02-01,0.065\n2001-03-01,0.353\n2001-04-01,0.915\n2001-05-01,2.06\n2001-06-01,8.395\n2001-07-01,7.262\n2001-08-01,6.026\n2001-09-01,6.761\n2001-10-01,1.531\n2001-11-01,0.152\n2001-12-01,0.171\n2002-01-01,0.417\n2002-02-01,0.129\n2002-03-01,0.059\n2002-04-01,0.352\n2002-05-01,1.827\n2002-06-01,7.207\n2002-07-01,5.876\n2002-08-01,3.566\n2002-09-01,10.186\n2002-10-01,2.185\n2002-11-01,2.13\n2002-12-01,0.041\n2003-01-01,0.194\n2003-02-01,0.083\n2003-03-01,0.07\n2003-04-01,0.533\n2003-05-01,1.598\n2003-06-01,9.753\n2003-07-01,4.355\n2003-08-01,6.764\n2003-09-01,9.745\n2003-10-01,3.491\n2003-11-01,0.325\n2003-12-01,0.071\n2004-01-01,0.57\n2004-02-01,0.173\n2004-03-01,0.195\n2004-04-01,0.812\n2004-05-01,2.335\n2004-06-01,7.486\n2004-07-01,6.488\n2004-08-01,6.215\n2004-09-01,7.197\n2004-10-01,3.137\n2004-11-01,0.254\n2004-12-01,0.022\n2005-01-01,0.172\n2005-02-01,0.052\n2005-03-01,0.248\n2005-04-01,0.353\n2005-05-01,0.973\n2005-06-01,3.3\n2005-07-01,8.394\n2005-08-01,6.632\n2005-09-01,4.722\n2005-10-01,1.819\n2005-11-01,0.136\n2005-12-01,0.086\n2006-01-01,0.2\n2006-02-01,0.044\n2006-03-01,0.094\n2006-04-01,0.514\n2006-05-01,2.26\n2006-06-01,5.395\n2006-07-01,6.682\n2006-08-01,7.477\n2006-09-01,6.741\n2006-10-01,3.404\n2006-11-01,1.041\n2006-12-01,0.053\n2007-01-01,0.166\n2007-02-01,0.076\n2007-03-01,0.119\n2007-04-01,0.373\n2007-05-01,2.163\n2007-06-01,6.073\n2007-07-01,6.693\n2007-08-01,7.522\n2007-09-01,6.359\n2007-10-01,2.634\n2007-11-01,0.18\n2007-12-01,0.021\n2008-01-01,0.109\n2008-02-01,0.034\n2008-03-01,0.023\n2008-04-01,0.655\n2008-05-01,1.662\n2008-06-01,7.126\n2008-07-01,8.885\n2008-08-01,7.086\n2008-09-01,7.099\n2008-10-01,1.788\n2008-11-01,0.083\n2008-12-01,0.063\n2009-01-01,0.244\n2009-02-01,0.15\n2009-03-01,0.171\n2009-04-01,0.31\n2009-05-01,3.252\n2009-06-01,5.16\n2009-07-01,4.628\n2009-08-01,6.209\n2009-09-01,9.549\n2009-10-01,2.74\n2009-11-01,0.676\n2009-12-01,0.109\n2010-01-01,0.685\n2010-02-01,0.46\n2010-03-01,0.112\n2010-04-01,0.47\n2010-05-01,0.711\n2010-06-01,5.378\n2010-07-01,10.209\n2010-08-01,8.673\n2010-09-01,8.67\n2010-10-01,0.231\n2010-11-01,0.336\n2010-12-01,0.015\n2011-01-01,0.043\n2011-02-01,0.007\n2011-03-01,0.039\n2011-04-01,0.647\n2011-05-01,1.805\n2011-06-01,4.206\n2011-07-01,9.773\n2011-08-01,6.719\n2011-09-01,4.789\n2011-10-01,2.292\n2011-11-01,0.304\n2011-12-01,0.003\n2012-01-01,0.111\n2012-02-01,0.465\n2012-03-01,0.243\n2012-04-01,0.345\n2012-05-01,1.764\n2012-06-01,5.897\n2012-07-01,8.111\n2012-08-01,8.112\n2012-09-01,4.477\n2012-10-01,2.153\n2012-11-01,0.349\n2012-12-01,0.021\n2013-01-01,0.196\n2013-02-01,0.018\n2013-03-01,0.154\n2013-04-01,0.33\n2013-05-01,2.893\n2013-06-01,6.568\n2013-07-01,5.87\n2013-08-01,6.04\n2013-09-01,9.177\n2013-10-01,2.312\n2013-11-01,1.559\n2013-12-01,0.028\n2014-01-01,0.055\n2014-02-01,0.02\n2014-03-01,0.018\n2014-04-01,0.598\n2014-05-01,3.648\n2014-06-01,7.896\n2014-07-01,6.222\n2014-08-01,4.742\n2014-09-01,9.363\n2014-10-01,3.791\n2014-11-01,0.341\n2014-12-01,0.195\n2015-01-01,0.22\n2015-02-01,0.093\n2015-03-01,0.559\n2015-04-01,0.386\n2015-05-01,3.571\n2015-06-01,5.174\n2015-07-01,4.322\n2015-08-01,5.808\n2015-09-01,4.94\n2015-10-01,2.407\n2015-11-01,1.18\n2015-12-01,0.078\n2016-01-01,0.774\n2016-02-01,0.004\n2016-03-01,1.202\n2016-04-01,1.269\n2016-05-01,2.101\n2016-06-01,8.511\n2016-07-01,7.974\n2016-08-01,7.807\n2016-09-01,6.136\n2016-10-01,1.604\n2016-11-01,1.6\n2016-12-01,0.015\n2017-01-01,0.005\n2017-02-01,0.003\n2017-03-01,0.206\n2017-04-01,0.361\n2017-05-01,2.881\n2017-06-01,7.677\n2017-07-01,7.464\n2017-08-01,10.681\n2017-09-01,11.457\n2017-10-01,2.77\n2017-11-01,0.064\n2017-12-01,0.017\n2018-01-01,0.159\n2018-02-01,0.116\n2018-03-01,0.143\n2018-04-01,1.462\n2018-05-01,3.57\n2018-06-01,10.038\n2018-07-01,1.847\n2018-08-01,8.724\n2018-09-01,7.064\n2018-10-01,5.465\n2018-11-01,1.651\n2018-12-01,0.061\n2019-01-01,0.128\n2019-02-01,0.017\n2019-03-01,0.1\n2019-04-01,0.121\n2019-05-01,1.214\n2019-06-01,8.611\n2019-07-01,6.702\n2019-08-01,6.163\n2019-09-01,6.46\n2019-10-01,5.387\n2019-11-01,1.496\n2019-12-01,0.084\n2020-01-01,0.18\n2020-02-01,0.135\n2020-03-01,0.234\n2020-04-01,0.174\n2020-05-01,0.88\n2020-06-01,4.486\n2020-07-01,6.704\n2020-08-01,8.412\n2020-09-01,9.374\n2020-10-01,0.768\n2020-11-01,0.063\n2020-12-01,0.033\n2021-01-01,0.259\n2021-02-01,0.006\n2021-03-01,0.186\n2021-04-01,0.382\n2021-05-01,3.332\n2021-06-01,8.092\n2021-07-01,5.136\n2021-08-01,7.831\n2021-09-01,9.484\n2021-10-01,2.716\n2021-11-01,0.248\n2021-12-01,0.016\n2022-01-01,0.335\n2022-02-01,0.015\n2022-03-01,0.06\n2022-04-01,0.82\n2022-05-01,0.697\n2022-06-01,5.486\n2022-07-01,5.556\n2022-08-01,6.51\n2022-09-01,7.467\n2022-10-01,2.751\n2022-11-01,0.352\n2022-12-01,0.058\n2023-01-01,0.014\n2023-02-01,0.012\n2023-03-01,0.055\n2023-04-01,0.298\n2023-05-01,3.176\n2023-06-01,1.071\n2023-07-01,5.963\n2023-08-01,6.479\n2023-09-01,3.798\n2023-10-01,3.315\n2023-11-01,3.617\n2023-12-01,0.141\n2024-01-01,0.115\n2024-02-01,0.225\n2024-03-01,0.01\n2024-04-01,0.151\n2024-05-01,0.861\n2024-06-01,4.33\n2024-07-01,11.396\n2024-08-01,7.33\n2024-09-01,10.859\n2024-10-01,3.516\n2024-11-01,0.317\n2024-12-01,0.053\n2025-01-01,0.806\n2025-02-01,0.231\n2025-03-01,0.08\n2025-04-01,0.092\n2025-05-01,4.559\n2025-06-01,9.224\n2025-07-01,5.576\n2025-08-01,7.118\n2025-09-01,9.99\n2025-10-01,4.41\n2025-11-01,0.141\n2025-12-01,0.089\n2026-01-01,0.047\n2026-02-01,0.056\n2026-03-01,0.343\n2026-04-01,1.295\n2026-05-01,1.837\n2026-06-01,9.287\n2026-07-01,5.377\n2026-08-01,2.937\n2026-09-01,4.407\n", "morelos_temperatura": "fecha,valor\n2000-01-01,18.245\n2000-02-01,19.694\n2000-03-01,21.699\n2000-04-01,22.981\n2000-05-01,22.208\n2000-06-01,20.367\n2000-07-01,20.675\n2000-08-01,20.425\n2000-09-01,20.376\n2000-10-01,19.946\n2000-11-01,20.342\n2000-12-01,18.149\n2001-01-01,18.277\n2001-02-01,19.674\n2001-03-01,20.461\n2001-04-01,22.853\n2001-05-01,22.444\n2001-06-01,20.529\n2001-07-01,20.135\n2001-08-01,20.27\n2001-09-01,19.606\n2001-10-01,19.17\n2001-11-01,18.194\n2001-12-01,18.563\n2002-01-01,18.42\n2002-02-01,19.007\n2002-03-01,21.67\n2002-04-01,23.299\n2002-05-01,23.713\n2002-06-01,21.436\n2002-07-01,20.438\n2002-08-01,20.918\n2002-09-01,19.911\n2002-10-01,19.837\n2002-11-01,17.867\n2002-12-01,17.877\n2003-01-01,17.808\n2003-02-01,19.959\n2003-03-01,21.518\n2003-04-01,23.259\n2003-05-01,24.366\n2003-06-01,20.982\n2003-07-01,20.606\n2003-08-01,20.599\n2003-09-01,20.147\n2003-10-01,19.553\n2003-11-01,19.015\n2003-12-01,16.885\n2004-01-01,17.122\n2004-02-01,18.393\n2004-03-01,21.093\n2004-04-01,21.626\n2004-05-01,22.269\n2004-06-01,20.682\n2004-07-01,20.565\n2004-08-01,20.982\n2004-09-01,20.085\n2004-10-01,19.971\n2004-11-01,19.057\n2004-12-01,18.13\n2005-01-01,17.822\n2005-02-01,19.76\n2005-03-01,21.235\n2005-04-01,23.816\n2005-05-01,23.356\n2005-06-01,23.946\n2005-07-01,21.151\n2005-08-01,20.438\n2005-09-01,20.307\n2005-10-01,20.037\n2005-11-01,19.394\n2005-12-01,19.017\n2006-01-01,18.889\n2006-02-01,19.807\n2006-03-01,21.566\n2006-04-01,23.464\n2006-05-01,22.459\n2006-06-01,21.328\n2006-07-01,20.977\n2006-08-01,20.449\n2006-09-01,20.357\n2006-10-01,20.191\n2006-11-01,18.212\n2006-12-01,18.047\n2007-01-01,18.569\n2007-02-01,19.481\n2007-03-01,21.329\n2007-04-01,22.557\n2007-05-01,23.035\n2007-06-01,21.612\n2007-07-01,20.566\n2007-08-01,20.29\n2007-09-01,19.7\n2007-10-01,18.941\n2007-11-01,18.697\n2007-12-01,19.242\n2008-01-01,18.031\n2008-02-01,20.279\n2008-03-01,20.891\n2008-04-01,23.175\n2008-05-01,22.901\n2008-06-01,20.869\n2008-07-01,19.741\n2008-08-01,20.156\n2008-09-01,19.642\n2008-10-01,19.203\n2008-11-01,18.072\n2008-12-01,18.55\n2009-01-01,19.027\n2009-02-01,19.433\n2009-03-01,20.94\n2009-04-01,23.115\n2009-05-01,22.435\n2009-06-01,21.749\n2009-07-01,21.225\n2009-08-01,21.408\n2009-09-01,19.93\n2009-10-01,20.194\n2009-11-01,18.314\n2009-12-01,18.477\n2010-01-01,17.006\n2010-02-01,17.219\n2010-03-01,20.74\n2010-04-01,22.557\n2010-05-01,24.542\n2010-06-01,22.87\n2010-07-01,20.344\n2010-08-01,20.279\n2010-09-01,19.942\n2010-10-01,19.067\n2010-11-01,18.247\n2010-12-01,17.199\n2011-01-01,18.741\n2011-02-01,19.688\n2011-03-01,21.219\n2011-04-01,23.64\n2011-05-01,24.145\n2011-06-01,22.147\n2011-07-01,19.963\n2011-08-01,20.604\n2011-09-01,19.758\n2011-10-01,19.132\n2011-11-01,19.534\n2011-12-01,18.789\n2012-01-01,18.565\n2012-02-01,19.045\n2012-03-01,21.187\n2012-04-01,22.457\n2012-05-01,22.977\n2012-06-01,21.63\n2012-07-01,20.068\n2012-08-01,20.229\n2012-09-01,20.015\n2012-10-01,20.437\n2012-11-01,18.556\n2012-12-01,19.379\n2013-01-01,18.752\n2013-02-01,20.77\n2013-03-01,20.275\n2013-04-01,23.393\n2013-05-01,22.968\n2013-06-01,21.455\n2013-07-01,20.469\n2013-08-01,20.554\n2013-09-01,19.716\n2013-10-01,19.724\n2013-11-01,18.807\n2013-12-01,18.644\n2014-01-01,17.685\n2014-02-01,20.675\n2014-03-01,21.929\n2014-04-01,23.253\n2014-05-01,21.965\n2014-06-01,20.803\n2014-07-01,20.327\n2014-08-01,20.878\n2014-09-01,20.155\n2014-10-01,19.515\n2014-11-01,18.59\n2014-12-01,18.236\n2015-01-01,18.468\n2015-02-01,19.106\n2015-03-01,20.141\n2015-04-01,23.283\n2015-05-01,22.231\n2015-06-01,20.975\n2015-07-01,21.01\n2015-08-01,21.473\n2015-09-01,20.203\n2015-10-01,20.599\n2015-11-01,20.72\n2015-12-01,19.892\n2016-01-01,18.249\n2016-02-01,19.145\n2016-03-01,20.556\n2016-04-01,23.726\n2016-05-01,24.623\n2016-06-01,21.368\n2016-07-01,21.123\n2016-08-01,20.973\n2016-09-01,20.443\n2016-10-01,20.411\n2016-11-01,19.371\n2016-12-01,20.135\n2017-01-01,19.17\n2017-02-01,20.135\n2017-03-01,21.537\n2017-04-01,22.94\n2017-05-01,24.029\n2017-06-01,21.659\n2017-07-01,20.431\n2017-08-01,21.03\n2017-09-01,19.982\n2017-10-01,19.812\n2017-11-01,19.351\n2017-12-01,18.418\n2018-01-01,17.443\n2018-02-01,20.251\n2018-03-01,22.585\n2018-04-01,22.854\n2018-05-01,22.972\n2018-06-01,21.142\n2018-07-01,21.652\n2018-08-01,20.535\n2018-09-01,20.46\n2018-10-01,20.207\n2018-11-01,19.546\n2018-12-01,18.599\n2019-01-01,19.011\n2019-02-01,21.117\n2019-03-01,22.255\n2019-04-01,23.288\n2019-05-01,24.512\n2019-06-01,22.456\n2019-07-01,20.997\n2019-08-01,21.415\n2019-09-01,20.94\n2019-10-01,20.091\n2019-11-01,20.144\n2019-12-01,19.521\n2020-01-01,19.107\n2020-02-01,20.82\n2020-03-01,22.737\n2020-04-01,24.544\n2020-05-01,23.869\n2020-06-01,22.707\n2020-07-01,21.546\n2020-08-01,20.801\n2020-09-01,20.394\n2020-10-01,19.791\n2020-11-01,19.833\n2020-12-01,19.386\n2021-01-01,19.438\n2021-02-01,20.147\n2021-03-01,22.208\n2021-04-01,23.502\n2021-05-01,22.856\n2021-06-01,20.853\n2021-07-01,20.903\n2021-08-01,20.848\n2021-09-01,20.083\n2021-10-01,20.647\n2021-11-01,19.217\n2021-12-01,20.259\n2022-01-01,19.102\n2022-02-01,19.937\n2022-03-01,21.727\n2022-04-01,23.862\n2022-05-01,24.59\n2022-06-01,21.908\n2022-07-01,22.103\n2022-08-01,21.161\n2022-09-01,20.395\n2022-10-01,20.537\n2022-11-01,20.836\n2022-12-01,19.013\n2023-01-01,19.582\n2023-02-01,20.405\n2023-03-01,22.396\n2023-04-01,23.412\n2023-05-01,23.797\n2023-06-01,25.578\n2023-07-01,22.67\n2023-08-01,21.981\n2023-09-01,22.07\n2023-10-01,21.294\n2023-11-01,20.748\n2023-12-01,18.749\n2024-01-01,20.124\n2024-02-01,21.348\n2024-03-01,23.514\n2024-04-01,24.694\n2024-05-01,26.689\n2024-06-01,24.365\n2024-07-01,21.671\n2024-08-01,21.602\n2024-09-01,21.053\n2024-10-01,19.791\n2024-11-01,20.919\n2024-12-01,19.564\n2025-01-01,19.634\n2025-02-01,21.576\n2025-03-01,23.115\n2025-04-01,23.479\n2025-05-01,24.034\n2025-06-01,20.904\n2025-07-01,21.06\n2025-08-01,21.494\n2025-09-01,21.189\n2025-10-01,20.599\n2025-11-01,20.31\n2025-12-01,20.044\n2026-01-01,19.43\n2026-02-01,20.374\n2026-03-01,21.308\n2026-04-01,23.199\n2026-05-01,24.303\n2026-06-01,21.778\n2026-07-01,22.186\n2026-08-01,22.817\n2026-09-01,22.219\n", "nortemx_humedad_suelo": "fecha,valor\n1982-01-01,0.2797\n1982-02-01,0.2752\n1982-03-01,0.2667\n1982-04-01,0.2584\n1982-05-01,0.251\n1982-06-01,0.2425\n1982-07-01,0.2692\n1982-08-01,0.2911\n1982-09-01,0.2786\n1982-10-01,0.2579\n1982-11-01,0.2458\n1982-12-01,0.2843\n1983-01-01,0.2829\n1983-02-01,0.2872\n1983-03-01,0.273\n1983-04-01,0.2615\n1983-05-01,0.2508\n1983-06-01,0.2421\n1983-07-01,0.2532\n1983-08-01,0.2973\n1983-09-01,0.2942\n1983-10-01,0.307\n1983-11-01,0.2913\n1983-12-01,0.2829\n1984-01-01,0.2769\n1984-02-01,0.276\n1984-03-01,0.2635\n1984-04-01,0.2535\n1984-05-01,0.245\n1984-06-01,0.257\n1984-07-01,0.3106\n1984-08-01,0.3362\n1984-09-01,0.3083\n1984-10-01,0.3031\n1984-11-01,0.2863\n1984-12-01,0.2981\n1985-01-01,0.3076\n1985-02-01,0.3026\n1985-03-01,0.2888\n1985-04-01,0.2786\n1985-05-01,0.2686\n1985-06-01,0.2633\n1985-07-01,0.2871\n1985-08-01,0.3142\n1985-09-01,0.3034\n1985-10-01,0.2958\n1985-11-01,0.2787\n1985-12-01,0.2747\n1986-01-01,0.2676\n1986-02-01,0.2659\n1986-03-01,0.2576\n1986-04-01,0.2496\n1986-05-01,0.2427\n1986-06-01,0.2462\n1986-07-01,0.3186\n1986-08-01,0.3191\n1986-09-01,0.3293\n1986-10-01,0.3045\n1986-11-01,0.2865\n1986-12-01,0.2947\n1987-01-01,0.2945\n1987-02-01,0.2854\n1987-03-01,0.2799\n1987-04-01,0.2708\n1987-05-01,0.2624\n1987-06-01,0.2593\n1987-07-01,0.2665\n1987-08-01,0.3112\n1987-09-01,0.2953\n1987-10-01,0.2696\n1987-11-01,0.2546\n1987-12-01,0.2561\n1988-01-01,0.2633\n1988-02-01,0.2584\n1988-03-01,0.2514\n1988-04-01,0.244\n1988-05-01,0.237\n1988-06-01,0.2319\n1988-07-01,0.2935\n1988-08-01,0.3179\n1988-09-01,0.2984\n1988-10-01,0.2676\n1988-11-01,0.2542\n1988-12-01,0.256\n1989-01-01,0.2698\n1989-02-01,0.27\n1989-03-01,0.2579\n1989-04-01,0.2479\n1989-05-01,0.2383\n1989-06-01,0.2306\n1989-07-01,0.2469\n1989-08-01,0.307\n1989-09-01,0.3043\n1989-10-01,0.2695\n1989-11-01,0.2545\n1989-12-01,0.2531\n1990-01-01,0.2639\n1990-02-01,0.2596\n1990-03-01,0.2533\n1990-04-01,0.2437\n1990-05-01,0.235\n1990-06-01,0.2341\n1990-07-01,0.2849\n1990-08-01,0.3304\n1990-09-01,0.3197\n1990-10-01,0.3172\n1990-11-01,0.2942\n1990-12-01,0.3008\n1991-01-01,0.3113\n1991-02-01,0.2999\n1991-03-01,0.2947\n1991-04-01,0.2835\n1991-05-01,0.2726\n1991-06-01,0.2624\n1991-07-01,0.2976\n1991-08-01,0.3273\n1991-09-01,0.334\n1991-10-01,0.3047\n1991-11-01,0.2905\n1991-12-01,0.3037\n1992-01-01,0.3272\n1992-02-01,0.3227\n1992-03-01,0.3095\n1992-04-01,0.2996\n1992-05-01,0.2876\n1992-06-01,0.2763\n1992-07-01,0.2804\n1992-08-01,0.3162\n1992-09-01,0.3001\n1992-10-01,0.2697\n1992-11-01,0.2612\n1992-12-01,0.2608\n1993-01-01,0.2932\n1993-02-01,0.283\n1993-03-01,0.2789\n1993-04-01,0.2656\n1993-05-01,0.2553\n1993-06-01,0.2466\n1993-07-01,0.3003\n1993-08-01,0.2989\n1993-09-01,0.3196\n1993-10-01,0.2909\n1993-11-01,0.2828\n1993-12-01,0.2769\n1994-01-01,0.274\n1994-02-01,0.2676\n1994-03-01,0.2604\n1994-04-01,0.2528\n1994-05-01,0.2451\n1994-06-01,0.2471\n1994-07-01,0.2582\n1994-08-01,0.2759\n1994-09-01,0.2825\n1994-10-01,0.2576\n1994-11-01,0.2535\n1994-12-01,0.2741\n1995-01-01,0.2751\n1995-02-01,0.2705\n1995-03-01,0.2623\n1995-04-01,0.2502\n1995-05-01,0.2403\n1995-06-01,0.2323\n1995-07-01,0.2561\n1995-08-01,0.2713\n1995-09-01,0.2908\n1995-10-01,0.2663\n1995-11-01,0.2552\n1995-12-01,0.2467\n1996-01-01,0.2437\n1996-02-01,0.2367\n1996-03-01,0.2303\n1996-04-01,0.224\n1996-05-01,0.2172\n1996-06-01,0.218\n1996-07-01,0.2717\n1996-08-01,0.2946\n1996-09-01,0.3146\n1996-10-01,0.2713\n1996-11-01,0.2553\n1996-12-01,0.2521\n1997-01-01,0.2475\n1997-02-01,0.2438\n1997-03-01,0.2392\n1997-04-01,0.2333\n1997-05-01,0.2276\n1997-06-01,0.2264\n1997-07-01,0.2493\n1997-08-01,0.2916\n1997-09-01,0.2822\n1997-10-01,0.2582\n1997-11-01,0.2469\n1997-12-01,0.2614\n1998-01-01,0.2627\n1998-02-01,0.2538\n1998-03-01,0.2471\n1998-04-01,0.2379\n1998-05-01,0.2288\n1998-06-01,0.2227\n1998-07-01,0.2546\n1998-08-01,0.2838\n1998-09-01,0.2663\n1998-10-01,0.2457\n1998-11-01,0.2441\n1998-12-01,0.2366\n1999-01-01,0.2308\n1999-02-01,0.2264\n1999-03-01,0.2206\n1999-04-01,0.2155\n1999-05-01,0.21\n1999-06-01,0.2114\n1999-07-01,0.2848\n1999-08-01,0.3006\n1999-09-01,0.282\n1999-10-01,0.2466\n1999-11-01,0.23\n1999-12-01,0.2235\n2000-01-01,0.2198\n2000-02-01,0.2154\n2000-03-01,0.2119\n2000-04-01,0.2077\n2000-05-01,0.2028\n2000-06-01,0.2177\n2000-07-01,0.2772\n2000-08-01,0.277\n2000-09-01,0.2619\n2000-10-01,0.2692\n2000-11-01,0.2763\n2000-12-01,0.2612\n2001-01-01,0.2554\n2001-02-01,0.2546\n2001-03-01,0.2493\n2001-04-01,0.2387\n2001-05-01,0.2293\n2001-06-01,0.2268\n2001-07-01,0.2692\n2001-08-01,0.2986\n2001-09-01,0.276\n2001-10-01,0.2684\n2001-11-01,0.2456\n2001-12-01,0.2408\n2002-01-01,0.2356\n2002-02-01,0.2503\n2002-03-01,0.2402\n2002-04-01,0.2304\n2002-05-01,0.2219\n2002-06-01,0.2183\n2002-07-01,0.2531\n2002-08-01,0.3002\n2002-09-01,0.2761\n2002-10-01,0.2519\n2002-11-01,0.24\n2002-12-01,0.2493\n2003-01-01,0.2489\n2003-02-01,0.2467\n2003-03-01,0.2439\n2003-04-01,0.2334\n2003-05-01,0.2244\n2003-06-01,0.2204\n2003-07-01,0.2473\n2003-08-01,0.2803\n2003-09-01,0.2826\n2003-10-01,0.2721\n2003-11-01,0.2532\n2003-12-01,0.2441\n2004-01-01,0.2588\n2004-02-01,0.2644\n2004-03-01,0.2564\n2004-04-01,0.2475\n2004-05-01,0.2369\n2004-06-01,0.2348\n2004-07-01,0.2646\n2004-08-01,0.2993\n2004-09-01,0.2964\n2004-10-01,0.2828\n2004-11-01,0.2887\n2004-12-01,0.2904\n2005-01-01,0.2942\n2005-02-01,0.2984\n2005-03-01,0.284\n2005-04-01,0.2711\n2005-05-01,0.2597\n2005-06-01,0.2505\n2005-07-01,0.2564\n2005-08-01,0.3094\n2005-09-01,0.2885\n2005-10-01,0.2617\n2005-11-01,0.2449\n2005-12-01,0.2396\n2006-01-01,0.2354\n2006-02-01,0.2306\n2006-03-01,0.2255\n2006-04-01,0.2204\n2006-05-01,0.2146\n2006-06-01,0.2132\n2006-07-01,0.2672\n2006-08-01,0.3166\n2006-09-01,0.3226\n2006-10-01,0.2875\n2006-11-01,0.2751\n2006-12-01,0.2698\n2007-01-01,0.2745\n2007-02-01,0.2769\n2007-03-01,0.2647\n2007-04-01,0.2549\n2007-05-01,0.2461\n2007-06-01,0.2438\n2007-07-01,0.2752\n2007-08-01,0.3158\n2007-09-01,0.3081\n2007-10-01,0.2861\n2007-11-01,0.2622\n2007-12-01,0.2759\n2008-01-01,0.2678\n2008-02-01,0.2649\n2008-03-01,0.2536\n2008-04-01,0.2441\n2008-05-01,0.2361\n2008-06-01,0.2297\n2008-07-01,0.2842\n2008-08-01,0.3152\n2008-09-01,0.3277\n2008-10-01,0.2919\n2008-11-01,0.2741\n2008-12-01,0.2697\n2009-01-01,0.2657\n2009-02-01,0.2608\n2009-03-01,0.2542\n2009-04-01,0.2479\n2009-05-01,0.2408\n2009-06-01,0.2394\n2009-07-01,0.2854\n2009-08-01,0.3049\n2009-09-01,0.2952\n2009-10-01,0.2878\n2009-11-01,0.2683\n2009-12-01,0.2605\n2010-01-01,0.2612\n2010-02-01,0.2819\n2010-03-01,0.2686\n2010-04-01,0.2572\n2010-05-01,0.2477\n2010-06-01,0.2397\n2010-07-01,0.2678\n2010-08-01,0.3073\n2010-09-01,0.3022\n2010-10-01,0.2846\n2010-11-01,0.2621\n2010-12-01,0.2532\n2011-01-01,0.2497\n2011-02-01,0.2448\n2011-03-01,0.2381\n2011-04-01,0.2314\n2011-05-01,0.2253\n2011-06-01,0.2194\n2011-07-01,0.2553\n2011-08-01,0.2934\n2011-09-01,0.2718\n2011-10-01,0.2429\n2011-11-01,0.2335\n2011-12-01,0.2407\n2012-01-01,0.2365\n2012-02-01,0.2298\n2012-03-01,0.2231\n2012-04-01,0.2163\n2012-05-01,0.2102\n2012-06-01,0.2062\n2012-07-01,0.2577\n2012-08-01,0.2959\n2012-09-01,0.2969\n2012-10-01,0.2722\n2012-11-01,0.2515\n2012-12-01,0.2463\n2013-01-01,0.2501\n2013-02-01,0.2442\n2013-03-01,0.2363\n2013-04-01,0.228\n2013-05-01,0.2209\n2013-06-01,0.2167\n2013-07-01,0.2747\n2013-08-01,0.303\n2013-09-01,0.3123\n2013-10-01,0.283\n2013-11-01,0.2692\n2013-12-01,0.2703\n2014-01-01,0.2654\n2014-02-01,0.2559\n2014-03-01,0.2488\n2014-04-01,0.2414\n2014-05-01,0.2337\n2014-06-01,0.2322\n2014-07-01,0.2728\n2014-08-01,0.3093\n2014-09-01,0.3199\n2014-10-01,0.2928\n2014-11-01,0.2754\n2014-12-01,0.2663\n2015-01-01,0.263\n2015-02-01,0.2745\n2015-03-01,0.2684\n2015-04-01,0.2593\n2015-05-01,0.2506\n2015-06-01,0.247\n2015-07-01,0.3033\n2015-08-01,0.3125\n2015-09-01,0.315\n2015-10-01,0.3009\n2015-11-01,0.2828\n2015-12-01,0.2755\n2016-01-01,0.2722\n2016-02-01,0.2633\n2016-03-01,0.254\n2016-04-01,0.2461\n2016-05-01,0.2385\n2016-06-01,0.2386\n2016-07-01,0.2807\n2016-08-01,0.3176\n2016-09-01,0.3232\n2016-10-01,0.2938\n2016-11-01,0.2726\n2016-12-01,0.2668\n2017-01-01,0.276\n2017-02-01,0.2701\n2017-03-01,0.2616\n2017-04-01,0.2522\n2017-05-01,0.244\n2017-06-01,0.2368\n2017-07-01,0.2783\n2017-08-01,0.3201\n2017-09-01,0.293\n2017-10-01,0.2697\n2017-11-01,0.251\n2017-12-01,0.2519\n2018-01-01,0.2508\n2018-02-01,0.2463\n2018-03-01,0.2427\n2018-04-01,0.2352\n2018-05-01,0.2282\n2018-06-01,0.2335\n2018-07-01,0.2772\n2018-08-01,0.3108\n2018-09-01,0.3166\n2018-10-01,0.3051\n2018-11-01,0.2964\n2018-12-01,0.2907\n2019-01-01,0.2852\n2019-02-01,0.2789\n2019-03-01,0.2744\n2019-04-01,0.265\n2019-05-01,0.2547\n2019-06-01,0.2473\n2019-07-01,0.2611\n2019-08-01,0.2901\n2019-09-01,0.3\n2019-10-01,0.2979\n2019-11-01,0.2853\n2019-12-01,0.303\n2020-01-01,0.289\n2020-02-01,0.2821\n2020-03-01,0.2809\n2020-04-01,0.271\n2020-05-01,0.2591\n2020-06-01,0.2489\n2020-07-01,0.2654\n2020-08-01,0.2648\n2020-09-01,0.272\n2020-10-01,0.2416\n2020-11-01,0.2258\n2020-12-01,0.2205\n2021-01-01,0.2232\n2021-02-01,0.2291\n2021-03-01,0.2223\n2021-04-01,0.2156\n2021-05-01,0.2098\n2021-06-01,0.2065\n2021-07-01,0.2736\n2021-08-01,0.2897\n2021-09-01,0.292\n2021-10-01,0.2591\n2021-11-01,0.2397\n2021-12-01,0.2335\n2022-01-01,0.2475\n2022-02-01,0.2395\n2022-03-01,0.231\n2022-04-01,0.2231\n2022-05-01,0.216\n2022-06-01,0.2178\n2022-07-01,0.266\n2022-08-01,0.3186\n2022-09-01,0.3307\n2022-10-01,0.2986\n2022-11-01,0.2801\n2022-12-01,0.2719\n2023-01-01,0.2681\n2023-02-01,0.2659\n2023-03-01,0.2627\n2023-04-01,0.2547\n2023-05-01,0.2464\n2023-06-01,0.2393\n2023-07-01,0.2456\n2023-08-01,0.2551\n2023-09-01,0.2438\n2023-10-01,0.2361\n2023-11-01,0.2298\n2023-12-01,0.2229\n2024-01-01,0.2208\n2024-02-01,0.226\n2024-03-01,0.2212\n2024-04-01,0.2153\n2024-05-01,0.2088\n2024-06-01,0.203\n2024-07-01,0.2475\n2024-08-01,0.2612\n2024-09-01,0.2436\n2024-10-01,0.2158\n2024-11-01,0.2042\n2024-12-01,0.1995\n2025-01-01,0.1966\n2025-02-01,0.1946\n2025-03-01,0.1913\n2025-04-01,0.1881\n2025-05-01,0.1851\n2025-06-01,0.1867\n2025-07-01,0.2607\n2025-08-01,0.2694\n2025-09-01,0.2939\n2025-10-01,0.2865\n2025-11-01,0.2612\n2025-12-01,0.2514\n2026-01-01,0.2518\n2026-02-01,0.2514\n2026-03-01,0.2396\n2026-04-01,0.2322\n2026-05-01,0.2237\n2026-06-01,0.2313\n2026-07-01,0.2556\n2026-08-01,0.2636\n2026-09-01,0.2716\n", "sahel_humedad_suelo": "fecha,valor\n1982-01-01,0.2724\n1982-02-01,0.2666\n1982-03-01,0.2615\n1982-04-01,0.2579\n1982-05-01,0.2578\n1982-06-01,0.2662\n1982-07-01,0.2812\n1982-08-01,0.316\n1982-09-01,0.309\n1982-10-01,0.2876\n1982-11-01,0.2719\n1982-12-01,0.2632\n1983-01-01,0.2575\n1983-02-01,0.2526\n1983-03-01,0.2481\n1983-04-01,0.2449\n1983-05-01,0.244\n1983-06-01,0.2543\n1983-07-01,0.2774\n1983-08-01,0.2976\n1983-09-01,0.2899\n1983-10-01,0.2689\n1983-11-01,0.2515\n1983-12-01,0.243\n1984-01-01,0.2377\n1984-02-01,0.2334\n1984-03-01,0.2298\n1984-04-01,0.2273\n1984-05-01,0.2273\n1984-06-01,0.2403\n1984-07-01,0.2584\n1984-08-01,0.27\n1984-09-01,0.2647\n1984-10-01,0.2668\n1984-11-01,0.2484\n1984-12-01,0.2397\n1985-01-01,0.2344\n1985-02-01,0.2301\n1985-03-01,0.2266\n1985-04-01,0.2241\n1985-05-01,0.2231\n1985-06-01,0.2266\n1985-07-01,0.269\n1985-08-01,0.3065\n1985-09-01,0.3062\n1985-10-01,0.2826\n1985-11-01,0.2642\n1985-12-01,0.2547\n1986-01-01,0.2484\n1986-02-01,0.2432\n1986-03-01,0.2385\n1986-04-01,0.2354\n1986-05-01,0.2347\n1986-06-01,0.2398\n1986-07-01,0.2727\n1986-08-01,0.2994\n1986-09-01,0.3075\n1986-10-01,0.2943\n1986-11-01,0.2745\n1986-12-01,0.2642\n1987-01-01,0.2571\n1987-02-01,0.2511\n1987-03-01,0.2458\n1987-04-01,0.2422\n1987-05-01,0.2404\n1987-06-01,0.2548\n1987-07-01,0.2689\n1987-08-01,0.2967\n1987-09-01,0.2956\n1987-10-01,0.2783\n1987-11-01,0.2605\n1987-12-01,0.2509\n1988-01-01,0.2448\n1988-02-01,0.2398\n1988-03-01,0.2354\n1988-04-01,0.2326\n1988-05-01,0.2317\n1988-06-01,0.2381\n1988-07-01,0.2781\n1988-08-01,0.3262\n1988-09-01,0.3314\n1988-10-01,0.3074\n1988-11-01,0.2889\n1988-12-01,0.2786\n1989-01-01,0.2714\n1989-02-01,0.2651\n1989-03-01,0.2596\n1989-04-01,0.2556\n1989-05-01,0.254\n1989-06-01,0.2582\n1989-07-01,0.2908\n1989-08-01,0.3267\n1989-09-01,0.324\n1989-10-01,0.3054\n1989-11-01,0.2889\n1989-12-01,0.2792\n1990-01-01,0.2723\n1990-02-01,0.2664\n1990-03-01,0.261\n1990-04-01,0.2572\n1990-05-01,0.2562\n1990-06-01,0.2581\n1990-07-01,0.288\n1990-08-01,0.3094\n1990-09-01,0.2975\n1990-10-01,0.2792\n1990-11-01,0.2624\n1990-12-01,0.253\n1991-01-01,0.247\n1991-02-01,0.2422\n1991-03-01,0.238\n1991-04-01,0.2351\n1991-05-01,0.2397\n1991-06-01,0.2554\n1991-07-01,0.2791\n1991-08-01,0.3179\n1991-09-01,0.314\n1991-10-01,0.2929\n1991-11-01,0.2766\n1991-12-01,0.2671\n1992-01-01,0.2609\n1992-02-01,0.2555\n1992-03-01,0.2506\n1992-04-01,0.2473\n1992-05-01,0.2466\n1992-06-01,0.2558\n1992-07-01,0.2822\n1992-08-01,0.3138\n1992-09-01,0.3188\n1992-10-01,0.2923\n1992-11-01,0.2752\n1992-12-01,0.2657\n1993-01-01,0.2593\n1993-02-01,0.2539\n1993-03-01,0.2491\n1993-04-01,0.2456\n1993-05-01,0.2444\n1993-06-01,0.2486\n1993-07-01,0.2784\n1993-08-01,0.3035\n1993-09-01,0.3072\n1993-10-01,0.2833\n1993-11-01,0.2666\n1993-12-01,0.2569\n1994-01-01,0.2504\n1994-02-01,0.2453\n1994-03-01,0.2407\n1994-04-01,0.2376\n1994-05-01,0.2372\n1994-06-01,0.2454\n1994-07-01,0.2804\n1994-08-01,0.332\n1994-09-01,0.3414\n1994-10-01,0.3242\n1994-11-01,0.3085\n1994-12-01,0.299\n1995-01-01,0.292\n1995-02-01,0.2857\n1995-03-01,0.2797\n1995-04-01,0.2754\n1995-05-01,0.2738\n1995-06-01,0.2766\n1995-07-01,0.2938\n1995-08-01,0.3277\n1995-09-01,0.3243\n1995-10-01,0.3045\n1995-11-01,0.2875\n1995-12-01,0.2778\n1996-01-01,0.2707\n1996-02-01,0.2644\n1996-03-01,0.2591\n1996-04-01,0.2553\n1996-05-01,0.2543\n1996-06-01,0.2634\n1996-07-01,0.2773\n1996-08-01,0.3073\n1996-09-01,0.3123\n1996-10-01,0.2937\n1996-11-01,0.2759\n1996-12-01,0.2656\n1997-01-01,0.2586\n1997-02-01,0.2529\n1997-03-01,0.2482\n1997-04-01,0.2449\n1997-05-01,0.2445\n1997-06-01,0.2586\n1997-07-01,0.2804\n1997-08-01,0.3061\n1997-09-01,0.3072\n1997-10-01,0.2868\n1997-11-01,0.2699\n1997-12-01,0.26\n1998-01-01,0.2535\n1998-02-01,0.2481\n1998-03-01,0.2434\n1998-04-01,0.2402\n1998-05-01,0.2404\n1998-06-01,0.2494\n1998-07-01,0.2748\n1998-08-01,0.3178\n1998-09-01,0.3311\n1998-10-01,0.3086\n1998-11-01,0.2902\n1998-12-01,0.2796\n1999-01-01,0.2722\n1999-02-01,0.266\n1999-03-01,0.2604\n1999-04-01,0.2564\n1999-05-01,0.2549\n1999-06-01,0.2575\n1999-07-01,0.2914\n1999-08-01,0.3422\n1999-09-01,0.3461\n1999-10-01,0.3281\n1999-11-01,0.3117\n1999-12-01,0.302\n2000-01-01,0.2952\n2000-02-01,0.2885\n2000-03-01,0.2824\n2000-04-01,0.2783\n2000-05-01,0.2765\n2000-06-01,0.2806\n2000-07-01,0.3002\n2000-08-01,0.3261\n2000-09-01,0.3172\n2000-10-01,0.2983\n2000-11-01,0.2811\n2000-12-01,0.2712\n2001-01-01,0.2644\n2001-02-01,0.2588\n2001-03-01,0.2538\n2001-04-01,0.2503\n2001-05-01,0.2499\n2001-06-01,0.2557\n2001-07-01,0.2884\n2001-08-01,0.3177\n2001-09-01,0.3152\n2001-10-01,0.2911\n2001-11-01,0.2723\n2001-12-01,0.2625\n2002-01-01,0.2559\n2002-02-01,0.2506\n2002-03-01,0.246\n2002-04-01,0.2428\n2002-05-01,0.2417\n2002-06-01,0.2471\n2002-07-01,0.2737\n2002-08-01,0.3084\n2002-09-01,0.3051\n2002-10-01,0.2849\n2002-11-01,0.2678\n2002-12-01,0.2576\n2003-01-01,0.251\n2003-02-01,0.2457\n2003-03-01,0.2412\n2003-04-01,0.2381\n2003-05-01,0.2372\n2003-06-01,0.2546\n2003-07-01,0.2912\n2003-08-01,0.324\n2003-09-01,0.3274\n2003-10-01,0.3032\n2003-11-01,0.2875\n2003-12-01,0.2771\n2004-01-01,0.27\n2004-02-01,0.2639\n2004-03-01,0.2587\n2004-04-01,0.2552\n2004-05-01,0.2545\n2004-06-01,0.2623\n2004-07-01,0.2859\n2004-08-01,0.3156\n2004-09-01,0.3142\n2004-10-01,0.2886\n2004-11-01,0.2716\n2004-12-01,0.2617\n2005-01-01,0.2553\n2005-02-01,0.2501\n2005-03-01,0.2457\n2005-04-01,0.2425\n2005-05-01,0.2415\n2005-06-01,0.2556\n2005-07-01,0.295\n2005-08-01,0.3169\n2005-09-01,0.3196\n2005-10-01,0.2998\n2005-11-01,0.2808\n2005-12-01,0.2701\n2006-01-01,0.2629\n2006-02-01,0.257\n2006-03-01,0.252\n2006-04-01,0.2485\n2006-05-01,0.2474\n2006-06-01,0.254\n2006-07-01,0.2721\n2006-08-01,0.3123\n2006-09-01,0.32\n2006-10-01,0.301\n2006-11-01,0.282\n2006-12-01,0.2713\n2007-01-01,0.2644\n2007-02-01,0.2585\n2007-03-01,0.2533\n2007-04-01,0.2498\n2007-05-01,0.2492\n2007-06-01,0.2519\n2007-07-01,0.2772\n2007-08-01,0.3262\n2007-09-01,0.3326\n2007-10-01,0.304\n2007-11-01,0.2876\n2007-12-01,0.2774\n2008-01-01,0.2704\n2008-02-01,0.2645\n2008-03-01,0.2591\n2008-04-01,0.2554\n2008-05-01,0.2544\n2008-06-01,0.2585\n2008-07-01,0.2972\n2008-08-01,0.3258\n2008-09-01,0.3312\n2008-10-01,0.3117\n2008-11-01,0.2947\n2008-12-01,0.2848\n2009-01-01,0.2775\n2009-02-01,0.2713\n2009-03-01,0.2659\n2009-04-01,0.262\n2009-05-01,0.2609\n2009-06-01,0.265\n2009-07-01,0.2881\n2009-08-01,0.3182\n2009-09-01,0.3266\n2009-10-01,0.306\n2009-11-01,0.2915\n2009-12-01,0.2821\n2010-01-01,0.2747\n2010-02-01,0.2684\n2010-03-01,0.2631\n2010-04-01,0.2595\n2010-05-01,0.259\n2010-06-01,0.2685\n2010-07-01,0.2988\n2010-08-01,0.339\n2010-09-01,0.3489\n2010-10-01,0.3295\n2010-11-01,0.3146\n2010-12-01,0.3059\n2011-01-01,0.2991\n2011-02-01,0.2928\n2011-03-01,0.2869\n2011-04-01,0.2828\n2011-05-01,0.2814\n2011-06-01,0.2844\n2011-07-01,0.2981\n2011-08-01,0.3273\n2011-09-01,0.3286\n2011-10-01,0.3078\n2011-11-01,0.2904\n2011-12-01,0.28\n2012-01-01,0.273\n2012-02-01,0.2671\n2012-03-01,0.2617\n2012-04-01,0.2582\n2012-05-01,0.2578\n2012-06-01,0.2699\n2012-07-01,0.3095\n2012-08-01,0.3454\n2012-09-01,0.3499\n2012-10-01,0.3309\n2012-11-01,0.3169\n2012-12-01,0.3081\n2013-01-01,0.3013\n2013-02-01,0.295\n2013-03-01,0.2891\n2013-04-01,0.2851\n2013-05-01,0.2833\n2013-06-01,0.2849\n2013-07-01,0.304\n2013-08-01,0.3342\n2013-09-01,0.3363\n2013-10-01,0.3162\n2013-11-01,0.3001\n2013-12-01,0.2905\n2014-01-01,0.2836\n2014-02-01,0.2773\n2014-03-01,0.2718\n2014-04-01,0.2679\n2014-05-01,0.2683\n2014-06-01,0.2729\n2014-07-01,0.2879\n2014-08-01,0.3236\n2014-09-01,0.3234\n2014-10-01,0.307\n2014-11-01,0.2893\n2014-12-01,0.2791\n2015-01-01,0.2723\n2015-02-01,0.2664\n2015-03-01,0.2612\n2015-04-01,0.2576\n2015-05-01,0.2559\n2015-06-01,0.2585\n2015-07-01,0.2867\n2015-08-01,0.3314\n2015-09-01,0.3393\n2015-10-01,0.3226\n2015-11-01,0.3066\n2015-12-01,0.2976\n2016-01-01,0.2914\n2016-02-01,0.2852\n2016-03-01,0.2793\n2016-04-01,0.2753\n2016-05-01,0.2744\n2016-06-01,0.2779\n2016-07-01,0.3079\n2016-08-01,0.3385\n2016-09-01,0.3353\n2016-10-01,0.314\n2016-11-01,0.299\n2016-12-01,0.2895\n2017-01-01,0.2825\n2017-02-01,0.2762\n2017-03-01,0.2706\n2017-04-01,0.2668\n2017-05-01,0.2667\n2017-06-01,0.2756\n2017-07-01,0.3083\n2017-08-01,0.3322\n2017-09-01,0.3278\n2017-10-01,0.3063\n2017-11-01,0.2886\n2017-12-01,0.2788\n2018-01-01,0.2721\n2018-02-01,0.2663\n2018-03-01,0.261\n2018-04-01,0.2573\n2018-05-01,0.2562\n2018-06-01,0.2658\n2018-07-01,0.3009\n2018-08-01,0.336\n2018-09-01,0.3445\n2018-10-01,0.3258\n2018-11-01,0.3101\n2018-12-01,0.3009\n2019-01-01,0.2939\n2019-02-01,0.2874\n2019-03-01,0.2816\n2019-04-01,0.2776\n2019-05-01,0.2762\n2019-06-01,0.2814\n2019-07-01,0.3026\n2019-08-01,0.3325\n2019-09-01,0.3427\n2019-10-01,0.3276\n2019-11-01,0.3119\n2019-12-01,0.3023\n2020-01-01,0.2953\n2020-02-01,0.2887\n2020-03-01,0.2826\n2020-04-01,0.2787\n2020-05-01,0.2769\n2020-06-01,0.2858\n2020-07-01,0.3137\n2020-08-01,0.3501\n2020-09-01,0.3607\n2020-10-01,0.3406\n2020-11-01,0.327\n2020-12-01,0.3195\n2021-01-01,0.3126\n2021-02-01,0.3062\n2021-03-01,0.3005\n2021-04-01,0.2965\n2021-05-01,0.2953\n2021-06-01,0.2979\n2021-07-01,0.3148\n2021-08-01,0.3489\n2021-09-01,0.3433\n2021-10-01,0.325\n2021-11-01,0.3118\n2021-12-01,0.3026\n2022-01-01,0.296\n2022-02-01,0.2898\n2022-03-01,0.2841\n2022-04-01,0.2803\n2022-05-01,0.2788\n2022-06-01,0.285\n2022-07-01,0.3117\n2022-08-01,0.3394\n2022-09-01,0.3503\n2022-10-01,0.3331\n2022-11-01,0.3188\n2022-12-01,0.3102\n2023-01-01,0.3036\n2023-02-01,0.2972\n2023-03-01,0.2914\n2023-04-01,0.2875\n2023-05-01,0.2877\n2023-06-01,0.2948\n2023-07-01,0.3155\n2023-08-01,0.3343\n2023-09-01,0.3334\n2023-10-01,0.3129\n2023-11-01,0.2963\n2023-12-01,0.2861\n2024-01-01,0.279\n2024-02-01,0.2728\n2024-03-01,0.2674\n2024-04-01,0.2638\n2024-05-01,0.2633\n2024-06-01,0.2718\n2024-07-01,0.3101\n2024-08-01,0.3549\n2024-09-01,0.3638\n2024-10-01,0.3469\n2024-11-01,0.3333\n2024-12-01,0.3259\n2025-01-01,0.3194\n2025-02-01,0.3131\n2025-03-01,0.3072\n2025-04-01,0.303\n2025-05-01,0.3017\n2025-06-01,0.3022\n2025-07-01,0.3207\n2025-08-01,0.3399\n2025-09-01,0.3386\n2025-10-01,0.3179\n2025-11-01,0.3023\n2025-12-01,0.2933\n2026-01-01,0.2865\n2026-02-01,0.2805\n2026-03-01,0.2753\n2026-04-01,0.2718\n2026-05-01,0.2709\n2026-06-01,0.2802\n2026-07-01,0.295\n2026-08-01,0.3143\n2026-09-01,0.3065\n"}
def demo(nombre):
    return pd.read_csv(io.StringIO(DEMOS[nombre]), parse_dates=['fecha']).set_index('fecha')['valor'].asfreq('MS')
print('Casos disponibles:', ', '.join(DEMOS))

## 🧩 3. Casos resueltos

### Caso 1 · "La primavera eterna se calienta" — Morelos, México 🇲🇽
Morelos es famoso por su clima templado. ¿Sigue igual? Variable: **temperatura del aire** (modelo FLDAS, 2000–2026), promedio sobre todo el estado.

In [ ]:
r_mor_t = analizar(demo("morelos_temperatura"), "Temperatura del aire", "°C", region="Morelos, México", color="#d9534f")

### Caso 2 · "La lluvia que no cambió" — Morelos, México
Mismo lugar, otra variable: **lluvia** (FLDAS, 2000–2026). Aquí aprendemos que **no encontrar una tendencia significativa también es un resultado** — y la herramienta nos dice cuántos años más necesitaríamos.

In [ ]:
r_mor_p = analizar(demo("morelos_lluvia"), "Lluvia", "mm/día", region="Morelos, México", color="#2c7fb8")

### Caso 3 · "Mismo planeta, direcciones opuestas" — Sahel 🌍 vs. Norte de México 🌎
Con el calentamiento global, se espera que **algunas regiones se humedezcan y otras se sequen**: el aire más caliente mueve más vapor de agua y reorganiza dónde llueve (la circulación de Hadley se expande hacia los polos).

- **Sahel (África, 12–17° N):** franja semiárida al sur del Sahara que sufrió grandes sequías en los 70–80.
- **Norte de México (Sonora–Chihuahua, 26–31° N):** borde del cinturón desértico subtropical, en "megasequía" con el suroeste de EE. UU.

Variable: **humedad del suelo 10–40 cm** (FLDAS, 1982–2026).

In [ ]:
r_sahel = analizar(demo("sahel_humedad_suelo"), "Humedad del suelo 10–40 cm", "m³/m³", region="Sahel (África)", color="#2c7fb8")
r_nmx   = analizar(demo("nortemx_humedad_suelo"), "Humedad del suelo 10–40 cm", "m³/m³", region="Norte de México", color="#d9534f")
comparar(r_sahel, r_nmx)

#### 🧠 Lo que descubrimos
- **Sahel:** la humedad del suelo **aumenta** de forma muy significativa: la región sigue recuperándose de las sequías de los 70–80 (el "reverdecimiento del Sahel").
- **Norte de México:** la humedad del suelo **disminuye** de forma muy significativa — el suelo se seca. 2020–2024 tiene los valores más bajos de los 45 años.
- **¿Por qué direcciones opuestas?** Ambas regiones están en los bordes del gran cinturón seco subtropical. Con el calentamiento, el monzón de África occidental se intensifica y lleva más lluvia al Sahel, mientras que la zona seca subtropical se expande hacia el norte de México y el suroeste de EE. UU., y el calor extra evapora más agua del suelo. **Un mismo proceso global — el calentamiento que reorganiza la circulación atmosférica — produce tendencias de signo contrario según la región.**
- 🔍 *Pista para seguir investigando:* revisa la **temperatura** en ambas regiones (¿sube en las dos?) y la **lluvia GPM** del Sahel.

## 🕵️ 4. Tu investigación
**Paso A — Elige la pista.** Escoge la variable y la región en el formulario y ejecuta la celda. Se genera un enlace a **NASA Giovanni** ya configurado.

In [ ]:
#@title 🔎 Paso A · ¿Qué y dónde quieres investigar?
variable = "🌡️ Calor · Temperatura del aire a 2 m" #@param ["🌡️ Calor · Temperatura del aire a 2 m", "🌡️ Calor · Temperatura del aire en superficie (de día)", "🌡️ Calor · Temperatura de la superficie del suelo", "🌧️ Lluvia · Lluvia (satélites GPM)", "🌧️ Lluvia · Lluvia (modelo, corregida)", "🏜️ Sequía y agua · Humedad del suelo (10–40 cm)", "🏜️ Sequía y agua · Evapotranspiración", "🏜️ Sequía y agua · Escurrimiento superficial", "💧 Humedad del aire · Humedad específica a 2 m", "🏭 Calidad del aire · Dióxido de nitrógeno (NO₂)", "🏭 Calidad del aire · Aerosoles (polvo, humo, smog)", "🏭 Calidad del aire · Monóxido de carbono en superficie", "☁️ Nubes y sol · Fracción de nubes", "☁️ Nubes y sol · Radiación solar en superficie", "🛡️ Ozono · Ozono total en la columna", "❄️ Nieve · Cobertura de nieve", "🌊 Océano · Temperatura del mar (superficie)", "🌊 Océano · Clorofila (fitoplancton)"]
region = "Ciudad de México" #@param ["Ciudad de México", "Morelos", "Monterrey", "Guadalajara", "Península de Yucatán", "Norte de México (Sonora–Chihuahua)", "Golfo de California (mar)", "Sahel (África)", "Amazonía", "Desierto del Sahara", "Delhi (India)", "Los Ángeles (EE. UU.)", "Ártico (Groenlandia)", "Otra (escribe abajo)"]
bbox_personalizado = "-99.4,19.1,-98.9,19.7" #@param {type:"string"}
inicio = "2005-01-01" #@param {type:"date"}
fin = "2025-12-31" #@param {type:"date"}

REGIONES = {"Ciudad de México": "-99.4,19.1,-98.9,19.7", "Morelos": "-99.5,18.33,-98.63,19.14", "Monterrey": "-100.5,25.5,-100.1,25.9",
 "Guadalajara": "-103.5,20.5,-103.2,20.8", "Península de Yucatán": "-90.5,18.5,-87,21.5", "Norte de México (Sonora–Chihuahua)": "-112,26,-104,31",
 "Golfo de California (mar)": "-113,25,-109.5,29", "Sahel (África)": "-15,12,20,17", "Amazonía": "-70,-10,-55,0", "Desierto del Sahara": "-5,20,25,28",
 "Delhi (India)": "76.8,28.3,77.6,28.9", "Los Ángeles (EE. UU.)": "-118.7,33.7,-117.8,34.3", "Ártico (Groenlandia)": "-50,70,-25,80"}
bbox = bbox_personalizado if region.startswith("Otra") else REGIONES[region]
nombre_var = variable.split(" · ", 1)[1]
var_key = next(k for k, v in CATALOGO.items() if v["nombre"] == nombre_var)
info = CATALOGO[var_key]
url = enlace_giovanni(var_key, bbox, inicio, fin)
display(Markdown(f"""
**Investigarás:** {info['nombre']} — *{info['defin']}*
**Fuente:** {info['fuente']} · disponible desde {info['desde']} · **Región:** {region} (`{bbox}`)

**Paso B — Consigue la evidencia en NASA Giovanni:**
1. 👉 **[Abrir Giovanni con tu búsqueda]({url})** (inicia sesión con tu cuenta gratuita **NASA Earthdata**).
2. Revisa que esté seleccionado *Time Series, Area-Averaged* y presiona **Plot Data**.
3. Cuando aparezca la gráfica, ve a **Downloads → CSV** y guarda el archivo.
4. Ejecuta la siguiente celda y sube ese CSV.
"""))

In [ ]:
#@title 📈 Paso C · Sube el CSV de Giovanni y obtén el veredicto
try:
    from google.colab import files
    subido = files.upload()
    ruta = list(subido.keys())[0]
except ImportError:
    ruta = input("Ruta al CSV de Giovanni: ")
s, var_id, meta = leer_giovanni(ruta)
s, nombre, unidad, info = preparar(s, var_id)
mi_resultado = analizar(s, nombre, unidad, region=region)

**Paso D (opcional) — Compara con otra región.** Repite los pasos A–C con la misma variable en otra región; luego ejecuta:
```python
otra = mi_resultado          # guarda el primero así antes de repetir
comparar(otra, mi_resultado)
```

## 📖 Glosario del detective
| Término | Significado sencillo |
|---|---|
| **Serie de tiempo** | Una variable medida una y otra vez en el tiempo (aquí, cada mes). |
| **Tendencia** | Cambio sostenido en una dirección (sube o baja) a lo largo de los años. |
| **Ciclo estacional** | El sube y baja que se repite cada año por las estaciones. Hay que quitarlo para ver la tendencia. |
| **Anomalía** | Cuánto se aleja un mes de lo normal para ese mes. |
| **Pendiente / velocidad de cambio** | Cuánto cambia la variable por década. |
| **p-valor** | Probabilidad de ver un patrón así *por casualidad* si en realidad no hubiera tendencia. Menor a 0.05 → **significativa**. |
| **Intervalo de confianza 95 %** | Rango donde probablemente está la verdadera velocidad de cambio. Si no toca el cero, la tendencia es real. |
| **Prueba *t*** | Prueba estadística clásica para saber si la pendiente es distinta de cero. |
| **Mann-Kendall** | Prueba que solo pregunta "¿los valores tienden a ser mayores con el tiempo?", sin suponer una línea recta. |
| **Autocorrelación** | Los meses seguidos se parecen; ignorarlo haría parecer las tendencias más seguras de lo que son. |
| **Satélite vs. modelo** | El satélite *mide* desde el espacio; el modelo *combina* mediciones con física para llenar huecos. |

**Fuentes:** NASA GES DISC Giovanni · FLDAS (McNally et al., 2017) · MERRA-2 (Gelaro et al., 2017) · GPM IMERG · MODIS · AIRS · OMI. Métodos: Santer et al. (2000) para n efectivo; Weatherhead et al. (1998) para años necesarios; Hirsch et al. (1982) Mann-Kendall estacional.

*Equipo Terra · NASA Space Apps Challenge 2026*